### Cleaning Mid Cap Fund NAV Data

In [2]:
import pandas as pd
import numpy as np
import os

In [3]:
# Load raw combined dataset
df_all = pd.read_csv('combined_midcap_nav.csv')
df_all['date'] = pd.to_datetime(df_all['date'])

# Filter positive valid NAVs
df_all = df_all[df_all['nav'] > 0].copy()

funds = {
    'Axis Midcap Fund': 'axis_midcap',
    'Bandhan Midcap Fund': 'bandhan_midcap',
    'Edelweiss Mid Cap Fund': 'edelweiss_midcap',
    'Kotak Emerging Equity Fund': 'kotak_emerging_equity',
    'Mahindra Manulife Mid Cap Fund': 'mahindra_manulife_midcap'
}

In [4]:
def clean_nav_data(fund_label, df_source):
    df = df_source[df_source['fund_label'] == fund_label].copy()
    df = df[['date', 'nav', 'scheme_name']].rename(columns={'date': 'Date', 'nav': 'NAV'})
    
    # Study period: 2022-01-01 to 2025-12-31
    df = df[(df['Date'] >= '2022-01-01') & (df['Date'] <= '2025-12-31')]
    
    # Drop missing and sort chronologically
    df = df.dropna(subset=['Date', 'NAV']).sort_values('Date').reset_index(drop=True)
    return df

In [5]:
cleaned_data = {}
for fund_name in funds:
    cleaned_data[fund_name] = clean_nav_data(fund_name, df_all)

print('Cleaning completed for all 5 mid cap funds.')

Cleaning completed for all 5 mid cap funds.


In [6]:
for fund_name, df in cleaned_data.items():
    print('\n' + '=' * 60)
    print(fund_name)
    print('=' * 60)
    print('Rows:', len(df))
    print('Start date:', df['Date'].min().date())
    print('End date:', df['Date'].max().date())
    print('Missing dates:', df['Date'].isna().sum())
    print('Missing NAV:', df['NAV'].isna().sum())
    print('Start NAV:', df['NAV'].iloc[0])
    print('End NAV:', df['NAV'].iloc[-1])


Axis Midcap Fund
Rows: 989
Start date: 2022-01-03
End date: 2025-12-31
Missing dates: 0
Missing NAV: 0
Start NAV: 79.09
End NAV: 132.07

Bandhan Midcap Fund
Rows: 834
Start date: 2022-08-19
End date: 2025-12-31
Missing dates: 0
Missing NAV: 0
Start NAV: 10.011
End NAV: 18.252

Edelweiss Mid Cap Fund
Rows: 985
Start date: 2022-01-03
End date: 2025-12-31
Missing dates: 0
Missing NAV: 0
Start NAV: 59.553
End NAV: 106.845

Kotak Emerging Equity Fund
Rows: 952
Start date: 2022-01-03
End date: 2025-12-31
Missing dates: 0
Missing NAV: 0
Start NAV: 23.641
End NAV: 33.452

Mahindra Manulife Mid Cap Fund
Rows: 984
Start date: 2022-01-03
End date: 2025-12-31
Missing dates: 0
Missing NAV: 0
Start NAV: 19.0035
End NAV: 38.4986


## Saving Cleaned Datasets

In [8]:
os.makedirs('Mid_cap/clean', exist_ok=True)

for fund_name, df in cleaned_data.items():
    filename = funds[fund_name]
    output_path = f'Mid_cap/clean/{filename}.csv'
    df.to_csv(output_path, index=False)
    print(f'Saved: {output_path}')

Saved: Mid_cap/clean/axis_midcap.csv
Saved: Mid_cap/clean/bandhan_midcap.csv
Saved: Mid_cap/clean/edelweiss_midcap.csv
Saved: Mid_cap/clean/kotak_emerging_equity.csv
Saved: Mid_cap/clean/mahindra_manulife_midcap.csv


### Create Daily Returns and Drawdowns

In [10]:
analysis_data = {}
os.makedirs('Mid_cap/analysis', exist_ok=True)

for fund_name, df in cleaned_data.items():
    df_an = df.copy()
    df_an['Daily_Return'] = df_an['NAV'].pct_change()
    df_an['Running_Max'] = df_an['NAV'].cummax()
    df_an['Drawdown'] = (df_an['NAV'] - df_an['Running_Max']) / df_an['Running_Max']
    
    filename = funds[fund_name]
    path = f'Mid_cap/analysis/{filename}_analysis.csv'
    df_an.to_csv(path, index=False)
    analysis_data[fund_name] = df_an
    print(f'Saved analysis data: {path}')

Saved analysis data: Mid_cap/analysis/axis_midcap_analysis.csv
Saved analysis data: Mid_cap/analysis/bandhan_midcap_analysis.csv
Saved analysis data: Mid_cap/analysis/edelweiss_midcap_analysis.csv
Saved analysis data: Mid_cap/analysis/kotak_emerging_equity_analysis.csv
Saved analysis data: Mid_cap/analysis/mahindra_manulife_midcap_analysis.csv


### Calculate CAGR, Volatility, Max Drawdown and Sharpe Ratio

In [12]:
TRADING_DAYS = 252
RISK_FREE_RATE = 0.06
daily_rf = (1 + RISK_FREE_RATE) ** (1 / TRADING_DAYS) - 1

results = []
for fund_name, df in analysis_data.items():
    start_date = df['Date'].iloc[0]
    end_date = df['Date'].iloc[-1]
    start_nav = df['NAV'].iloc[0]
    end_nav = df['NAV'].iloc[-1]
    
    years = (end_date - start_date).days / 365.25
    cagr = ((end_nav / start_nav) ** (1 / years)) - 1
    
    daily_rets = df['Daily_Return'].dropna()
    volatility = daily_rets.std() * np.sqrt(TRADING_DAYS)
    max_dd = df['Drawdown'].min()
    
    excess_returns = daily_rets - daily_rf
    sharpe = (excess_returns.mean() / daily_rets.std()) * np.sqrt(TRADING_DAYS)
    
    results.append({
        'Fund': fund_name,
        'Start Date': start_date.strftime('%Y-%m-%d'),
        'End Date': end_date.strftime('%Y-%m-%d'),
        'Observations': len(df),
        'Start NAV': round(start_nav, 4),
        'End NAV': round(end_nav, 4),
        'CAGR': cagr,
        'Volatility': volatility,
        'Maximum Drawdown': max_dd,
        'Sharpe Ratio': sharpe
    })

mid_cap_results = pd.DataFrame(results)
output_path = 'Mid_cap/analysis/mid_cap_fund_results.csv'
mid_cap_results.to_csv(output_path, index=False)
print(f'Mid cap results saved to {output_path}')
print(mid_cap_results)

Mid cap results saved to Mid_cap/analysis/mid_cap_fund_results.csv
                             Fund  Start Date  ... Maximum Drawdown  Sharpe Ratio
0                Axis Midcap Fund  2022-01-03  ...        -0.195165      0.581231
1             Bandhan Midcap Fund  2022-08-19  ...        -0.227126      0.928845
2          Edelweiss Mid Cap Fund  2022-01-03  ...        -0.181717      0.703249
3      Kotak Emerging Equity Fund  2022-01-03  ...        -0.242741      0.291873
4  Mahindra Manulife Mid Cap Fund  2022-01-03  ...        -0.209212      0.842859

[5 rows x 10 columns]
